<a href="https://colab.research.google.com/github/saachikayadav/collusion-detection/blob/main/RAG_With_Knowledge_graph(Neo4j).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# langchain-core

contains simple, core abstractions that have emerged as a standard, as well as LangChain Expression Language as a way to compose these components together. This package is now at version 0.1 and all breaking changes will be accompanied by a minor version bump.

# langchain-community
contains all third party integrations. We will work with partners on splitting key integrations out into standalone packages over the next month.

# langchain
contains higher-level and use-case specific chains, agents, and retrieval algorithms that are at the core of your application's cognitive architecture. We are targeting a launch of a stable 0.1 release for langchain in early January.#

In [ ]:
%pip install --upgrade --quiet  langchain langchain-community langchain-openai langchain-experimental neo4j wikipedia tiktoken yfiles_jupyter_graphs

In [ ]:
!pip install -U langchain-neo4j

In [ ]:
!pip show langchain-neo4j

In [ ]:
from langchain_core.runnables import (
    RunnableBranch,
    RunnableLambda,
    RunnableParallel,
    RunnablePassthrough,
)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts.prompt import PromptTemplate

In [ ]:
from google.colab import userdata
OPENAI_API_KEY=userdata.get('OPENAI_API_KEY')

In [ ]:
from typing import Tuple, List, Optional

In [ ]:
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.output_parsers import StrOutputParser

In [ ]:
from langchain_core.runnables import ConfigurableField

In [ ]:
from yfiles_jupyter_graphs import GraphWidget
from neo4j import GraphDatabase


In [ ]:
import os

In [ ]:
try:
  import google.colab
  from google.colab import output
  output.enable_custom_widget_manager()
except:
  pass

In [ ]:
!pip install -U langchain-neo4j

In [ ]:
from langchain_neo4j import Neo4jVector

In [ ]:
from google.colab import userdata
OPENAI_API_KEY=userdata.get('OPENAI_API_KEY')

In [ ]:
NEO4J_URI="neo4j+s://31ca99ea.databases.neo4j.io"
NEO4J_USERNAME="31ca99ea"
NEO4J_PASSWORD="VD11G_grAzgKd7-4ZZT_aIdabNqylBZnJoAP_gUnV4c"

In [ ]:
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
os.environ["NEO4J_URI"] = NEO4J_URI
os.environ["NEO4J_USERNAME"] = NEO4J_USERNAME
os.environ["NEO4J_PASSWORD"] = NEO4J_PASSWORD

In [ ]:
from langchain_community.graphs import Neo4jGraph

In [ ]:
print("URI:", NEO4J_URI)
print("USERNAME:", NEO4J_USERNAME)

In [ ]:
from langchain_neo4j import Neo4jGraph
graph = Neo4jGraph(
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    database="31ca99ea",
    refresh_schema=False
)

In [ ]:
from google.colab import files

uploaded = files.upload()



TypeError: 'NoneType' object is not subscriptable

In [ ]:
import pandas as pd

# Get the name of the uploaded file
file_name = list(uploaded.keys())[0]
df = pd.read_excel("claims.xlsx")
df.head()

In [ ]:
from langchain_core.documents import Document

documents = []

for _, row in df.iterrows():
    text = " ".join(str(x) for x in row.values)
    documents.append(Document(page_content=text))

In [ ]:
from google.colab import userdata

OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY")

In [ ]:
from langchain_text_splitters import TokenTextSplitter
text_splitter = TokenTextSplitter(chunk_size=512, chunk_overlap=24)
documents = text_splitter.split_documents(documents[:3])

In [ ]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(
    model="meta-llama/llama-3.2-3b-instruct:free",
    api_key=OPENROUTER_API_KEY,  # your sk-or-v1 key
    base_url="https://openrouter.ai/api/v1",
    temperature=0
)

In [ ]:
from google.colab import userdata

OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY")

print("Loaded:", OPENROUTER_API_KEY is not None)
print("Starts with sk-or-v1:", OPENROUTER_API_KEY.startswith("sk-or-v1-") if OPENROUTER_API_KEY else None)
print("Length:", len(OPENROUTER_API_KEY) if OPENROUTER_API_KEY else None)

In [ ]:
from langchain_community.graphs import Neo4jGraph

graph = Neo4jGraph(
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    database="31ca99ea",
)

In [ ]:
from neo4j import GraphDatabase

driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USERNAME, NEO4J_PASSWORD)
)

In [ ]:
from neo4j import GraphDatabase
import pandas as pd

# Connect to Neo4j
driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USERNAME, NEO4J_PASSWORD)
)

# Start with 100 rows for testing
sample_df = df.head(100)

# Replace NaN with empty strings
sample_df = sample_df.fillna("")

# Convert dataframe to list of dictionaries
records = sample_df.to_dict("records")

query = """
UNWIND $rows AS row

MERGE (c:Claim {
    claimNumber: toString(row.ClaimNumber)
})

MERGE (p:Policy {
    policyNo: toString(row.PolicyNo)
})

MERGE (i:Insured {
    name: toString(row.InsuredName)
})

MERGE (city:City {
    name: toString(row.LossCity)
})

MERGE (state:State {
    name: toString(row.LossState)
})

MERGE (s:Surveyor {
    name: toString(row.SurveyorName)
})

MERGE (i)-[:HAS_POLICY]->(p)
MERGE (p)-[:HAS_CLAIM]->(c)
MERGE (c)-[:OCCURRED_IN]->(city)
MERGE (city)-[:IN_STATE]->(state)
MERGE (c)-[:SURVEYED_BY]->(s)

SET c.lossType = row.LossType,
    c.claimedAmount = row.ClaimedAmount,
    c.lossDate = row.DateOfLoss
"""

with driver.session() as session:
    session.run(query, rows=records)

print(f"Loaded {len(records)} rows successfully!")

In [ ]:
with driver.session() as session:
    session.run("MATCH (n) DETACH DELETE n")

print("Cleared old graph")

In [ ]:
# Clear old graph
with driver.session() as session:
    session.run("MATCH (n) DETACH DELETE n")

# Start with 100 rows first
records = df.head(100).fillna("").astype(str).to_dict("records")

query = """
UNWIND $rows AS row

MERGE (claim:Claim {claimNumber: row.ClaimNumber})
SET claim.isReferredForInvestigation = row.IsReferredForInvestigation,
    claim.manufacturingYear = row.ManufacturingYear,
    claim.causeOfLoss = row.CauseOfLoss,
    claim.damageDescription = row.DamageDescription,
    claim.claimStatus = row.ClaimStatus

MERGE (insurer:Insured {name: row.InsuredName})
MERGE (insurerCity:City {name: row.City})
MERGE (insurerState:State {name: row.State})

MERGE (lossCity:City {name: row.LossCity})
MERGE (lossState:State {name: row.LossState})

MERGE (surveyor:Surveyor {name: row.SurveyorName})
SET surveyor.id = row.SurveyorId

MERGE (surveyorCity:City {name: row.SurveyorCity})
MERGE (surveyorState:State {name: row.SurveyorState})

MERGE (police:PoliceStation {name: row.FIRPoliceStation})

MERGE (vehicle:Vehicle {
    registrationNumber: row.RegistrationNumber
})
SET vehicle.make = row.Make,
    vehicle.model = row.Model,
    vehicle.manufacturingYear = row.ManufacturingYear,
    vehicle.type = row.VehicleType

MERGE (cause:CauseOfLoss {name: row.CauseOfLoss})
MERGE (damage:DamageDescription {description: row.DamageDescription})

MERGE (insurer)-[:FILED]->(claim)

MERGE (insurer)-[:LIVES_IN]->(insurerCity)
MERGE (insurerCity)-[:IN_STATE]->(insurerState)

MERGE (claim)-[:OCCURRED_IN]->(lossCity)
MERGE (lossCity)-[:IN_STATE]->(lossState)

MERGE (claim)-[:SURVEYED_BY]->(surveyor)
MERGE (surveyor)-[:BASED_IN]->(surveyorCity)
MERGE (surveyorCity)-[:IN_STATE]->(surveyorState)

MERGE (claim)-[:REPORTED_TO]->(police)

MERGE (claim)-[:INVOLVES_VEHICLE]->(vehicle)

MERGE (claim)-[:CAUSED_BY]->(cause)
MERGE (claim)-[:HAS_DAMAGE]->(damage)
"""

with driver.session() as session:
    session.run(query, rows=records)

print("Loaded structured claim graph with surveyor/state/city/police/investigation/vehicle/loss relationships")

In [ ]:
# directly show the graph resulting from the given Cypher query
default_cypher = "MATCH (s)-[r:!MENTIONS]->(t) RETURN s,r,t LIMIT 50"

In [ ]:
from yfiles_jupyter_graphs import GraphWidget
from neo4j import GraphDatabase

In [ ]:
def showGraph(cypher: str = default_cypher):
    # create a neo4j session to run queries
    driver = GraphDatabase.driver(
        uri = os.environ["NEO4J_URI"],
        auth = (os.environ["NEO4J_USERNAME"],
                os.environ["NEO4J_PASSWORD"]))
    session = driver.session()
    widget = GraphWidget(graph = session.run(cypher).graph())
    widget.node_label_mapping = 'id'
    display(widget)
    return widget

In [ ]:
try:
  import google.colab
  from google.colab import output
  output.enable_custom_widget_manager()
except:
  pass

In [ ]:
showGraph("""
MATCH (n)-[r]->(m)
RETURN n,r,m
LIMIT 100
""")

In [ ]:
from typing import Tuple, List, Optional

In [ ]:
from langchain_neo4j import Neo4jVector

In [ ]:
!pip install -q langchain-huggingface

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5"
)

vector_index = Neo4jVector.from_existing_graph(
    embeddings,
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    database="31ca99ea",
    node_label="__Entity__",
    text_node_properties=["name", "description"],
    embedding_node_property="embedding"
)

In [ ]:
cypher = """
MERGE (person:InsuredPerson {name: $InsuredName})

MERGE (policy:Policy {policyNo: $PolicyNo})
SET policy.insurerName = $InsuredName

MERGE (person)-[:HAS_POLICY]->(policy)

MERGE (policyType:PolicyType {name: $PolicyType})
MERGE (policy)-[:HAS_POLICY_TYPE]->(policyType)

MERGE (city:City {name: $City})
MERGE (state:State {name: $State})
MERGE (city)-[:IN_STATE]->(state)
MERGE (person)-[:LOCATED_IN]->(city)

MERGE (endorsement:EndorsementType {name: $EndorsementType})
MERGE (policy)-[:HAS_ENDORSEMENT]->(endorsement)

MERGE (payment:PaymentMode {name: $ModeofPayment})
MERGE (policy)-[:PAID_BY]->(payment)

MERGE (branch:Branch {code: $BranchCode})
MERGE (policy)-[:ISSUED_AT]->(branch)

MERGE (claim:Claim {claimNumber: $ClaimNumber})
SET claim.isReferredForInvestigation = $IsReferredForInvestigation

MERGE (person)-[:FILED_CLAIM]->(claim)
MERGE (policy)-[:HAS_CLAIM]->(claim)

MERGE (lossType:LossType {name: $LossType})
MERGE (claim)-[:HAS_LOSS_TYPE]->(lossType)

MERGE (lossCity:City {name: $LossCity})
MERGE (lossState:State {name: $LossState})
MERGE (lossCity)-[:IN_STATE]->(lossState)
MERGE (claim)-[:OCCURRED_IN]->(lossCity)

MERGE (police:PoliceStation {name: $FIRPoliceStation})
MERGE (claim)-[:REPORTED_TO]->(police)

MERGE (vehicle:Vehicle {
    registrationNumber: $RegistrationNumber
})
SET vehicle.manufacturingYear = $ManufacturingYear

MERGE (vehicleType:VehicleType {name: $VehicleType})
MERGE (vehicle)-[:HAS_VEHICLE_TYPE]->(vehicleType)
MERGE (claim)-[:INVOLVES_VEHICLE]->(vehicle)

MERGE (surveyor:Surveyor {name: $SurveyorName})
MERGE (surveyorCity:City {name: $SurveyorCity})
MERGE (surveyorState:State {name: $SurveyorState})
MERGE (surveyorCity)-[:IN_STATE]->(surveyorState)
MERGE (surveyor)-[:LOCATED_IN]->(surveyorCity)
MERGE (claim)-[:SURVEYED_BY]->(surveyor)

MERGE (garage:Garage {name: $GarageName})
SET garage.isActive = $IsActive

MERGE (garageCity:City {name: $GarageCity})
MERGE (garage)-[:LOCATED_IN]->(garageCity)
MERGE (claim)-[:REPAIRED_AT]->(garage)
"""

In [ ]:
from pydantic import BaseModel, Field

In [ ]:
from pydantic import BaseModel, Field
# Extract entities from text
class Entities(BaseModel):
    """Identifying information about entities."""

    names: List[str] = Field(
        ...,
        description="All the person, organization, or business entities that "
        "appear in the text",
    )


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts.prompt import PromptTemplate

In [ ]:
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are extracting organization and person entities from the text.",
        ),
        (
            "human",
            "Use the given format to extract information from the following "
            "input: {question}",
        ),
    ]
)

In [ ]:
entity_chain = prompt | llm.with_structured_output(Entities)

In [ ]:
entity_chain.invoke({
    "question": "Show me claims handled in Delhi"
})

In [ ]:
import re

def remove_lucene_chars(text):
    return re.sub(r'[+\-&|!(){}\[\]^"~*?:\\/]', ' ', text)

In [ ]:
def generate_full_text_query(input: str) -> str:
    full_text_query = ""
    words = [el for el in remove_lucene_chars(input).split() if el]
    for word in words[:-1]:
        full_text_query += f" {word}~2 AND"
    full_text_query += f" {words[-1]}~2"
    return full_text_query.strip()


In [ ]:
# Fulltext index query
def structured_retriever(question: str) -> str:
    q = question.lower()

    with driver.session() as session:

        # 1. Police station with most accidents
        if "police station" in q and ("most" in q or "highest" in q):
            response = session.run("""
            MATCH (c:Claim)-[:REPORTED_TO]->(p:PoliceStation)
            RETURN p.name AS policeStation, count(c) AS accidents
            ORDER BY accidents DESC
            LIMIT 10
            """).data()

            return "\n".join(
                [f"{r['policeStation']} -> {r['accidents']} claims" for r in response]
            )

        # 2. Claims in a city
        if "delhi" in q:
            city = "Delhi"
            response = session.run("""
            MATCH (c:Claim)-[:OCCURRED_IN]->(city:City)
            WHERE toLower(city.name) = toLower($city)
            RETURN c.claimNumber AS claimNumber, city.name AS city
            LIMIT 20
            """, city=city).data()

            return "\n".join(
                [f"Claim {r['claimNumber']} occurred in {r['city']}" for r in response]
            )

        # 3. Surveyor with most claims
        if "surveyor" in q and ("most" in q or "highest" in q):
            response = session.run("""
            MATCH (c:Claim)-[:SURVEYED_BY]->(s:Surveyor)
            RETURN s.name AS surveyor, count(c) AS claims
            ORDER BY claims DESC
            LIMIT 10
            """).data()

            return "\n".join(
                [f"{r['surveyor']} -> {r['claims']} claims" for r in response]
            )

        # 4. Investigation-related claims
        if "investigation" in q:
            response = session.run("""
            MATCH (c:Claim)
            WHERE toLower(c.isReferredForInvestigation) IN ["true", "yes", "1", "y"]
            RETURN c.claimNumber AS claimNumber,
                   c.causeOfLoss AS causeOfLoss,
                   c.damageDescription AS damageDescription
            LIMIT 20
            """).data()

            return "\n".join(
                [
                    f"Claim {r['claimNumber']} | Cause: {r['causeOfLoss']} | Damage: {r['damageDescription']}"
                    for r in response
                ]
            )

        return "I don't have a matching Cypher template for this question yet."

In [ ]:
 print(structured_retriever("Which Police station has most accidents?"))

In [ ]:
def retriever(question: str):
    print(f"Search query: {question}")
    structured_data = structured_retriever(question)
    unstructured_data = [el.page_content for el in vector_index.similarity_search(question)]
    final_data = f"""Structured data:
{structured_data}
Unstructured data:
{"#Document ". join(unstructured_data)}
    """
    return final_data

In [ ]:
_template = """Given the following conversation and a follow up question, rephrase the follow up question to be a standalone question,
in its original language.
Chat History:
{chat_history}
Follow Up Input: {question}
Standalone question:"""

In [ ]:
CONDENSE_QUESTION_PROMPT = PromptTemplate.from_template(_template)

In [ ]:
def _format_chat_history(chat_history: List[Tuple[str, str]]) -> List:
    buffer = []
    for human, ai in chat_history:
        buffer.append(HumanMessage(content=human))
        buffer.append(AIMessage(content=ai))
    return buffer

In [ ]:
_search_query = RunnableBranch(
    # If input includes chat_history, we condense it with the follow-up question
    (
        RunnableLambda(lambda x: bool(x.get("chat_history"))).with_config(
            run_name="HasChatHistoryCheck"
        ),  # Condense follow-up question and chat into a standalone_question
        RunnablePassthrough.assign(
            chat_history=lambda x: _format_chat_history(x["chat_history"])
        )
        | CONDENSE_QUESTION_PROMPT
        | ChatOpenAI(temperature=0)
        | StrOutputParser(),
    ),
    # Else, we have no chat history, so just pass through the question
    RunnableLambda(lambda x : x["question"]),
)

In [ ]:
template = """Answer the question based only on the following context:
{context}

Question: {question}
Use natural language and be concise.
Answer:"""

In [ ]:
prompt = ChatPromptTemplate.from_template(template)

In [ ]:
chain = (
    RunnableParallel(
        {
            "context": _search_query | retriever,
            "question": RunnablePassthrough(),
        }
    )
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
from langchain_core.runnables import RunnableLambda

def insurance_retriever(question):
    with driver.session() as session:

        if "state" in question.lower():

            rows = session.run("""
            MATCH (c:Claim)-[:OCCURRED_IN]->(:City)-[:IN_STATE]->(s:State)
            RETURN s.name AS state, count(c) AS cases
            ORDER BY cases DESC
            LIMIT 10
            """).data()

            return str(rows)

    return "No data found"

retriever = RunnableLambda(insurance_retriever)

In [ ]:
# 1. Get graph result using Cypher
with driver.session() as session:
    rows = session.run("""
    MATCH (c:Claim)-[:OCCURRED_IN]->(:City)-[:IN_STATE]->(s:State)
    RETURN s.name AS state, count(c) AS cases
    ORDER BY cases DESC
    LIMIT 10
    """).data()

print(rows)

# 2. Ask LLM to summarize the Cypher result
prompt = f"""
You are analyzing insurance claims data.

Question: Which state has the most cases?

Cypher result:
{rows}

Answer clearly in 2-3 sentences.
"""

llm.invoke(prompt)

In [ ]:
chain.invoke({
    "question": "Which state has the most cases?"
})